<a href="https://colab.research.google.com/github/minbj1226/pytorch-study/blob/main/09_Computer_Vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 이미지 데이터는 어떻게 표현되는가?

### 픽셀과 해상도는 무엇인가?

### RGB 이미지와 Grayscale 이미지의 차이는 무엇인가?

### 이미지의 channel은 무엇을 의미하는가?

### Convolution 연산이란 무엇인가?

### Kernel(Filter)는 어떤 역할을 하는가?

### Stride는 무엇이며 출력 크기에 어떤 영향을 주는가?

### Padding은 왜 사용하는가?

### Pooling은 무엇이며 왜 사용하는가?

### Feature Map이란 무엇인가?

### Receptive Field란 무엇인가?

### CNN에서 layer가 깊어질수록 어떤 특징을 보게 되는가?

# Convolutional Neural Networks
## 1. Zero-Padding
- 이미지의 높이와 너비 가장자리에 0을 추가하여, 컨볼루션 후 출력 크기 감소와 경계 정보 손실을 줄이는 기법

## 2. conv_single_step
- 입력의 한 영역과 필터 가중치를 곱해 합을 구한 뒤, 편향을 더해 하나의 출력값을 만드는 연산

## 3. conv_forward
- 필터를 입력 전체에 슬라이딩하며 각 위치에서 입력의 한 영역과 연산해 출력 feature map을 만드는 과정

## 4. pool_forward
- 입력의 일정 영역에서 최댓값 또는 평균값을 선택하여 공간 크기를 줄이고 주요 특징을 요약하는 과정

## 5. conv_backward
- 컨볼루션 층의 역전파 과정으로, 출력 gradient를 이용해 입력, 가중치, 편향에 대한 gradient를 계산하는 과정

## 6. create_mask_from_window
- max pooling 역전파에서 윈도우 내 최댓값의 위치만 True로 표시하는 마스크를 만드는 과정

## 7. distribute_value
- average pooling 역전파에서 출력 gradient 값을 윈도우의 모든 위치에 균등하게 나누어 분배하는 과정

## 8. pool_backward
- pooling 층의 역전파 과정으로, max pooling은 최댓값 위치만 gradient를 전달하고 average pooling은 전체에 균등 분배하는 과정

In [8]:
import numpy as np

# Zero-Padding 예시 - 높이와 너비 축에 3칸씩 zero padding 적용
def zero_pad(X, pad):
  X_pad = np.pad(X, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode='constant', constant_values=0)

  return X_pad

x = np.random.randn(4, 3, 3, 2)
x_pad = zero_pad(x, 3)
x_pad

In [10]:
# conv_single_step
def conv_single_step(a_slice_prev, W, b):
  s = a_slice_prev * W
  Z = np.sum(s)
  Z = Z + float(b)

  return Z

a_slice_prev = np.random.randn(4, 4, 3)
W = np.random.randn(4, 4, 3)
b = np.random.randn(1, 1, 1)

conv_single_step(a_slice_prev, W, b)

/tmp/ipykernel_3763/3697335542.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Z = Z + float(b)


np.float64(-9.909627442782588)

In [13]:
# conv_forward
def conv_forward(A_prev, W, b, hparameters):
  (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
  (f, f, n_C_prev, n_C) = W.shape

  stride = hparameters['stride']
  pad = hparameters['pad']

  n_H = int((n_H_prev - f + 2 * pad) / stride) + 1
  n_W = int((n_W_prev - f + 2 * pad) / stride) + 1

  Z = np.zeros((m, n_H, n_W, n_C))

  A_prev_pad = zero_pad(A_prev, pad)

  for i in range(m):
    a_prev_pad = A_prev_pad[i]
    for h in range(n_H):
      for w in range(n_W):
        for c in range(n_C):
          vert_start = h * stride
          vert_end = vert_start + f
          horiz_start = w * stride
          horiz_end = horiz_start + f

          a_slice_prev = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]
          weights = W[:, :, :, c]
          biases = b[:, :, :, c]
          Z[i, h, w, c] = conv_single_step(a_slice_prev, weights, biases)

  cache = (A_prev, W, b, hparameters)

  return Z, cache


np.random.seed(1)
A_prev = np.random.randn(2, 5, 7, 4)
W = np.random.randn(3, 3, 4, 8)
b = np.random.randn(1, 1, 1, 8)
hparameters = {'pad': 1, 'stride': 2}

Z, cache = conv_forward(A_prev, W, b, hparameters)
Z.shape

/tmp/ipykernel_3763/3697335542.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Z = Z + float(b)


(2, 3, 4, 8)

# Keras
## Sequential API vs Functional API
### Sequential API(순차적 모델)
- Layer들을 순차적으로 쌓는 방식(=일직선 구조)
- 단순한 모델 설계에 적합

### Functional API
- 복잡한 모델 구축에 더 유용
- Layer를 그래프 형태로 연결하여 유연함

In [1]:
# Sequential API 모델 예시
def model():
  model = Sequential([
      # 높이와 가로에 패딩을 3 적용, 입력 이미지의 높이: 64, 가로: 64, 채널: 3
      ZeroPadding2D(padding=(3, 3), input_shape=(64, 64, 3)),
      # 7x7 크기, stride 1의 필터 32개를 적용하여 출력 채널 32개의 feature map을 생성
      Conv2D(32, (7, 7), strides=(1, 1)),
      # axis=3인 채널에 해당하는 배치 정규화 적용
      BatchNormalization(axis=3),
      # ReLU 활성화 함수 적용
      ReLU(),
      # 최댓값 폴링 적용
      MaxPooling2D(),
      # 다차원 feature map을 1차원으로 펴기
      Flatten(),
      # 이진 분류를 위한 출력층
      Dense(1, activation='sigmoid')
  ])

  return model

In [2]:
# Functional API 모델 예시
def model(input_shape):
  #(64, 64, 3) 이미지 삽입
  input_img = input(shape=input_shape)
  # 4x4 크기, stride 1의 필터 8개를 적용하여 채널 8개의 feature map을 생성
  Z1 = Conv2D(8, (4, 4), strides=(1, 1), padding='same')(input_img)
  # ReLU 함수 적용
  A1 = ReLU()(Z1)
  # 최댓값 폴링 적용(8x8, stride가 8)
  P1 = MaxPool2D(pool_size=(8, 8), strides=(8, 8), padding='same')(A1)
  # 2x2 크기, stride 1의 필터 16개를 적용하여 채널 16개의 feature map을 생성
  Z2 = Conv2D(16, (2, 2), strides=(1, 1), padding='same')(P1)
  # ReLU 함수 적용
  A2 = ReLU()(Z2)
  # 최댓값 폴링 적용(4x4, stride가 4)
  P2 = MaxPool2D(pool_size=(4, 4), strides=(4, 4), padding='same')(A2)
  # 평탄화 작업
  F = Flatten()(P2)
  # 이진 분류 작업
  outputs = Dense(1, activation='sigmoid')(F)

  model = Model(inputs=input_img, outputs=outputs)

  return model